# data conversion scripts

converts everything to zarr files

performs hotpixel correction too

# SPAD512 single bin

In [ ]:
"""
Processes quanta SPAD512 data into a Zarr format. Saves events as normalized points in
space. Also performs hot pixel correction.
"""
%load_ext autoreload
%autoreload 2
%matplotlib inline
from pathlib import Path

from spadlib.io import read_quanta_bin, write_quanta_zarr
from spadlib.processing import correct_hotpixels

fps = 100_000
binpath = Path("data/quanta/lift_jump.bin")
quantadata_path = Path(f"data/quanta/{binpath.stem}.zarr")
print(f"Processing raw binary file {binpath} into binframes...")
quantaframes = read_quanta_bin(
    binpath, h=512, w=512
)
nframes = quantaframes.shape[0]
T_exp = nframes / fps
quantaframes = correct_hotpixels(quantaframes)
write_quanta_zarr(quantadata_path, quantaframes, T_exp=T_exp, save_coords=True)

# SPAD512 raw data folder

In [ ]:
"""
SPAD512 raw data folder (a bunch of .bin files instead of a single one) to Zarr format.
"""
%load_ext autoreload
%autoreload 2
%matplotlib inline
from pathlib import Path

import numpy as np

from spadlib.io import read_quanta_dir, write_quanta_zarr
from spadlib.processing import correct_hotpixels

datanames = ["data/quanta/opticalsetupsfm-prettydark", "data/quanta/opticalsetupsfm-slightlydark"]
for dataname in datanames:
    dataname = Path(dataname)
    print(f"Processing {dataname}...")
    frames = read_quanta_dir(dataname, H=512, W=512)

    out_path = dataname.parent / f"{dataname.name}.zarr"

    print("Correcting hotpixels...")
    frames = correct_hotpixels(frames)
    write_quanta_zarr(out_path, frames, fps=100_000)
    del frames

# npy file from the wision data

In [ ]:
"""
npy file from the wision data or something to Zarr format.
"""
%load_ext autoreload
%autoreload 2
%matplotlib inline
from pathlib import Path
import numpy as np

from spadlib.io import read_quanta_dir, write_quanta_zarr
from spadlib.processing import correct_hotpixels

dataname = "lighter"
data_path = Path(f"data/quanta/{dataname}.npy")
a = np.load(data_path)
A = np.unpackbits(a, axis=2)[:, ::-1, ::-1]

out_path = data_path.parent / f"{dataname}.zarr"

A_sliced = A[30000:96000, :, 16:]
A_sliced = correct_hotpixels(A_sliced)
write_quanta_zarr(out_path, A_sliced, fps=1 / 1.05e-5)

# Asynchronous SPAD data conversion

In [ ]:
"""
Asynchronous SPAD data conversion to Zarr format.
"""
%load_ext autoreload
%autoreload 2
%matplotlib inline
import numpy as np
from pathlib import Path

from spadlib.io import read_async_spad_dir, write_async_spad_zarr

T_exp = 0.1
W = 138
H = 91

datapath = Path("data/async-spad/AC_fan/G")
if not datapath.is_dir():
    raise ValueError()
outpath = datapath.parent / f"{datapath.name}.zarr"

points, pixel_timeseries = read_async_spad_dir(datapath, h=H, w=W, T_exp=T_exp, normalize=False)

write_async_spad_zarr(outpath, pixel_timeseries, T_exp=T_exp)